In [1]:
import os
os.environ["TRANSFORMERS_NO_TF"] = "1"
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainer, Seq2SeqTrainingArguments, EarlyStoppingCallback
from datasets import load_dataset, DatasetDict
import numpy as np
import evaluate

rouge = evaluate.load("rouge")

/home/stefu/anaconda3/envs/finalproject/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
raw_dataset = load_dataset('json', data_files='../datasets/article_summary/valid_finnish_articles_200.json')

original_train = raw_dataset['train']

train_valid = original_train.train_test_split(test_size=0.2, seed=123)

dataset = DatasetDict({
    'train': train_valid['train'],
    'validation': train_valid['test']
})

In [3]:
model_name = "google/mt5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

model.gradient_checkpointing_enable()

# Preprocessing function
def preprocess_function(examples):
    inputs = ["tiivistä: " + text for text in examples["text"]]

    model_inputs = tokenizer(
        inputs,
        max_length=512,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        examples["summary"],
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    labels["input_ids"] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in labels["input_ids"]
    ]

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
/home/stefu/anaconda3/envs/finalproject/lib/python3.10/site-packages/transformers/convert_slow_tokenizer.py:559: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


In [4]:
tokenized_dataset = dataset.map(preprocess_function, batched=True, remove_columns=dataset["train"].column_names)

def compute_metrics(pred):
    labels_ids = pred.label_ids
    pred_ids = pred.predictions

    pred_ids = np.where(pred_ids != -100, pred_ids, tokenizer.pad_token_id)
    labels_ids = np.where(labels_ids != -100, labels_ids, tokenizer.pad_token_id)

    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    labels_str = tokenizer.batch_decode(labels_ids, skip_special_tokens=True)

    # Compute ROUGE
    rouge_scores = rouge.compute(predictions=pred_str, references=labels_str)

    return {
        "rouge1": round(rouge_scores["rouge1"] * 100, 2),
        "rouge2": round(rouge_scores["rouge2"] * 100, 2),
        "rougeL": round(rouge_scores["rougeL"] * 100, 2),
    }

training_args = Seq2SeqTrainingArguments(
    logging_strategy="epoch",
    eval_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    weight_decay=0.01,
    save_strategy="epoch",
    save_total_limit=2,
    num_train_epochs=30,
    predict_with_generate=True,
    generation_max_length=128,
    generation_num_beams=4,
    metric_for_best_model="rouge2",
    greater_is_better=True,
)
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

trainer.train()

trainer.save_model("./finetuned-mt5-finnish-summarizer")

/tmp/ipykernel_106790/214704479.py:38: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
Using EarlyStoppingCallback without load_best_model_at_end=True. Once training is finished, the best model will not be loaded automatically.
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel
1,12.783700,5.509598,6.280000,3.020000,5.870000
2,9.210200,3.377642,10.370000,4.240000,9.090000
3,6.190000,2.802948,14.410000,5.790000,11.410000
4,4.544200,2.570256,12.660000,4.750000,11.420000
5,3.789400,2.534030,14.470000,5.370000,13.150000
6,3.500900,2.480969,14.590000,6.750000,13.700000
7,3.537100,2.497702,14.460000,5.800000,13.700000
8,3.226800,2.449547,16.350000,7.610000,15.440000
9,3.186900,2.451900,15.650000,7.460000,14.260000
10,3.130800,2.352077,20.140000,9.060000,18.060000


Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


In [5]:
from projects.summarizer.datascrape import scrape_article
from transformers import pipeline

summarizer = pipeline("text2text-generation", model="./finetuned-mt5-finnish-summarizer", tokenizer="google/mt5-base")

new_text = scrape_article("https://yle.fi/a/74-20158531", "yle")

input_text = "Tiivistä: " + new_text

summary = summarizer(input_text, max_length=100, min_length=50, do_sample=False)
print(summary[0])

/home/stefu/anaconda3/envs/finalproject/lib/python3.10/site-packages/transformers/convert_slow_tokenizer.py:559: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
Device set to use cuda:0


{'generated_text': '<extra_id_0>, mutta tunkeutumiset ovat lisääntyneet. Poliisi epäilee rikosoikeudellinen vastuuta 15 vuotiaana. Oulun poliisin mukaan tunkeutumiset ovat entistä useammin.'}
